### Part 1: Email Setup

In [7]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [8]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
EMAIL_PORT = os.getenv("EMAIL_PORT")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

if EMAIL_PORT:
    print("Email port is set")
else:
    print("Email port is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD and EMAIL_PORT

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

Email address is set
SMTP server is set
App password is set
Email port is set
Email is set up and we will try using it


In [9]:
# Here we go

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, EMAIL_PORT) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [10]:
send_email("Testing testing 123", 
            "Fingers crossed..", 
            "<html><body><strong>Fingers</strong> crossed..</body></html>")

In [11]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [12]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [13]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [ ]:
send_message(
    "Big news", 
    "Communications are a go!", 
    "<html><body>Communications are a <strong>go!</strong></body></html>"
)

### Part 2: Orchestrating by Code

In [16]:
intro = """
You are a sales agent working for FSS (Financial Software and Systems),
a company that provides FSS BLAZE™, a cloud-native payments platform for fintechs and payment service providers.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [25]:
sales_agent1 = Agent(
    name="Profressional Sales Agent",
    instructions=instructions1,
    model=MODEL_NAME
)

sales_agent2 = Agent(
    name="Humorous Sales Agent",
    instructions=instructions2,
    model=MODEL_NAME
)

sales_agent3 = Agent(
    name="Executive Sales Agent",
    instructions=instructions3,
    model=MODEL_NAME
)

In [26]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Modernize your payments stack with FSS BLAZE™

Hi [First Name],

I’m reaching out from FSS. We work with fintechs and payment service providers that need a more agile, cloud-native payments platform to support growth, speed, and operational resilience.

FSS BLAZE™ is designed to help organizations launch and scale payment services faster, with the flexibility to adapt to evolving market and regulatory demands. It supports modern payment processing needs while reducing the complexity of legacy infrastructure.

If improving time-to-market, scalability, and platform reliability is on your agenda this quarter, I’d welcome a brief conversation to see whether BLAZE™ could be relevant for your team.

Would you be open to a 15-minute call next week?

Best regards,  
[Your Name]  
FSS | Financial Software and Systems

In [28]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print("\n\n---\n\n")
    print(output)



---


Subject: Modernize Payments with a Cloud-Native Platform

Hi [First Name],

I’m reaching out from FSS, where we help fintechs and payment service providers build and scale modern payment operations with FSS BLAZE™, our cloud-native payments platform.

As payment volumes grow, many teams find that legacy infrastructure slows down innovation, increases operational complexity, and makes scaling more expensive than it should be. FSS BLAZE is designed to address those challenges with a flexible, resilient platform built for speed, scale, and reliability.

We work with organizations looking to:
- Launch and manage payment services faster
- Improve operational efficiency
- Scale securely with cloud-native architecture
- Adapt quickly to changing market and regulatory demands

If this is a priority for your team, I’d welcome a brief conversation to explore whether FSS BLAZE could be relevant to your roadmap.

Would you be open to a 15-minute call next week?

Best regards,  
[Your Name]

In [30]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a senior decision-maker at a fintech or payment service provider
receiving these emails about FSS BLAZE™, a cloud-native payments platform.
Pick the one you are most likely to respond to.

Judge each email on:
- Relevance: does it speak to a real pain point for a fintech or PSP?
- Credibility: does it sound trustworthy and specific, not hype?
- Clarity: is the message easy to grasp on a quick read?
- Call to action: is the next step clear and low-effort?
- Length: does it respect a busy reader's time?

Rules:
- Pick exactly one email.
- Do not merge, edit, or improve any email.
- Reply with the selected email reproduced word for word, and nothing else.
- Do not give an explanation.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)

In [32]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

emails = "Cold Sales Emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

best = await Runner.run(sales_picker, input=emails)

print(f"\n\n---\n\nBest email:\n\n{best.final_output}")



---

Best email:

Subject: Scaling your payments stack without adding complexity

Hi [First Name],

I’m reaching out from FSS, where we help fintechs and payment service providers modernize and scale their payments infrastructure with FSS BLAZE™ — our cloud-native payments platform.

If your team is looking to improve agility, resilience, and time-to-market across card and payment processing, BLAZE is designed to help you do that without the overhead of legacy infrastructure.

In brief, we support organizations that want to:
- launch and manage payment products faster
- scale securely in cloud environments
- reduce operational complexity across the payments stack

If this is on your roadmap, I’d welcome the opportunity to share a short overview and see whether it could be relevant for your team.

Would you be open to a brief call next week?

Best regards,  
[Your Name]  
[Title]  
FSS | Financial Software and Systems  
[Email] | [Phone]


In [33]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """Send an email to the preconfigured recipient.

    Use this tool when the user asks to send, email, or share information by email.
    Provide both a plain-text and an HTML version of the same message content.

    Args:
        subject: The email subject line. Keep it short and descriptive.
        text_body: The plain-text version of the email body, used by clients that
            cannot render HTML.
        html_body: The HTML version of the email body, with the same content as
            text_body. Use simple HTML tags (e.g. <p>, <b>, <ul>, <a>).

    Returns:
        A confirmation message indicating the email was sent.
    """
    send_message(subject, text_body,html_body)
    return "Email sent successfully"

In [35]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The email subject line. Keep it short and descriptive.',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The plain-text version of the email body, used by clients that\ncannot render HTML.',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML version of the email body, with the same content as\ntext_body. Use simple HTML tags (e.g. <p>, <b>, <ul>, <a>).',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [39]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a senior decision-maker at a fintech or payment service provider
receiving these emails about FSS BLAZE™, a cloud-native payments platform.
Pick the one you are most likely to respond to.

Judge each email on:
- Relevance: does it speak to a real pain point for a fintech or PSP?
- Credibility: does it sound trustworthy and specific, not hype?
- Clarity: is the message easy to grasp on a quick read?
- Call to action: is the next step clear and low-effort?
- Length: does it respect a busy reader's time?

Output format:
- Line 1: the selected email's subject line, starting with "Subject:", word for word.
- Line 2: leave blank.
- Line 3 onward: the selected email's body word for word, from the greeting to the signature.
- The subject line appears only once, on line 1. Do not repeat it anywhere in the body.

Rules:
- Pick exactly one email.
- Do not merge, edit, or improve any email.
- Reply with the selected email in the output format above, and nothing else.
- Do not give an explanation.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(
    name="Sales Email Sender", 
    instructions=decision, 
    model=MODEL_NAME,
    tools=[send_email_tool], 
    model_settings=require_tool
)


In [40]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

Final response:
Subject: Payments platform that won’t make your team cry

Hi [First Name],

If your payments stack feels a bit like a game of Jenga in a wind tunnel, we should talk.

FSS BLAZE™ is a cloud-native payments platform built for fintechs and payment service providers that want to scale without the usual chaos, duct tape, and late-night “why is this failing?” messages.

With BLAZE, you can:
- launch faster
- scale smarter
- keep operations and compliance from becoming a horror movie
- give your team a platform they don’t secretly resent

We help payment businesses modernize without turning every change into a six-month saga.

Would you be open to a quick chat next week? If not, I can also send a short overview and let you decide whether it’s worth a second glance.

Best,  
[Your Name]  
FSS | Financial Software and Systems


### Part 3a: Orchestrating by LLMs

In [42]:
description = "use this tool to write a sales email. In the input, just instruct it to write a sales email"

tool1 = sales_agent1.as_tool(
    name="Sales Email Writer", 
    description=description, 
)

tool1

TypeError: Agent.as_tool() got an unexpected keyword argument 'name'